In [4]:
import numpy as np
import pandas as pd
import random
import math

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from joblib import Parallel, delayed
from sklearn.preprocessing import LabelEncoder

In [6]:
# 加载数据集
data = pd.read_csv('breast_s.csv')

categorical_cols = data.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    label_encoder = LabelEncoder()
    data[col]=label_encoder.fit_transform(data[col])

y = data.iloc[:, -1]
X = data.iloc[:, :-1]


# 特征标准化
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 定义模型
models = {
    'SVM': LinearSVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB()
}

# 定义评价指标函数
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }

# 运行次数
n_runs = 20  # 您可以将其设置为 20

# 存储结果
results = {name: {'accuracy': [], 'precision': [], 'recall': [], 'f1': []} for name in models.keys()}

# 开始多次实验
def run_experiment(run):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random.randint(0, 42+n_runs+run))
    local_results = {name: {'accuracy': None, 'precision': None, 'recall': None, 'f1': None} for name in models.keys()}
    for name, model in models.items():
        metrics = evaluate_model(model, X_train, X_test, y_train, y_test)
        local_results[name] = metrics
    return local_results

# 使用并行计算运行多次实验
all_results = Parallel(n_jobs=-1)(delayed(run_experiment)(run) for run in range(n_runs))

# 汇总结果
for result in all_results:
    for name in models.keys():
        for metric_name in ['accuracy', 'precision', 'recall', 'f1']:
            results[name][metric_name].append(result[name][metric_name])

# 计算平均值和标准差
for name in models.keys():
    print(f'\nModel: {name}')
    for metric_name in ['accuracy', 'precision', 'recall', 'f1']:
        scores = results[name][metric_name]
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        print(f'{metric_name.capitalize()}: {mean_score:.4f} ± {std_score:.4f}')


Model: SVM
Accuracy: 0.7681 ± 0.0565
Precision: 0.7517 ± 0.0822
Recall: 0.7681 ± 0.0565
F1: 0.7449 ± 0.0725

Model: Decision Tree
Accuracy: 0.7043 ± 0.0646
Precision: 0.7094 ± 0.0652
Recall: 0.7043 ± 0.0646
F1: 0.7018 ± 0.0669

Model: KNN
Accuracy: 0.7422 ± 0.0544
Precision: 0.7263 ± 0.0680
Recall: 0.7422 ± 0.0544
F1: 0.7200 ± 0.0647

Model: Naive Bayes
Accuracy: 0.7724 ± 0.0458
Precision: 0.7888 ± 0.0474
Recall: 0.7724 ± 0.0458
F1: 0.7763 ± 0.0453


In [2]:
from twdgranule import run_model

# 加载数据
data = pd.read_csv('banknote.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

# 设置运行次数
n_runs = 10  # 或者 20，根据您的需要

# 设置 delta 和 alpha
m = X.shape[1]
delta = 0.05 * math.sqrt(m)
alpha = 0.75  # 您可以根据需要调整 alpha 值

# 存储结果
accuracies = []
precisions = []
recalls = []
f1_scores = []
coverages = []

for run in range(n_runs):
    # 随机划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random.randint(0, 42+n_runs+run))  # 设置 random_state 以确保可重复性

    # 调用模型
    accuracy, precision, recall, f1, coverage = run_model(X_train, y_train, X_test, y_test, delta, alpha)

    # 存储结果
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)
    coverages.append(coverage)

# 计算平均值和标准差
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

mean_recall = np.mean(recalls)
std_recall = np.std(recalls)

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_coverage = np.mean(coverages)
std_coverage = np.std(coverages)

# 输出结果
print(f'Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}')
print(f'Precision: {mean_precision:.4f} ± {std_precision:.4f}')
print(f'Recall: {mean_recall:.4f} ± {std_recall:.4f}')
print(f'F1-score: {mean_f1:.4f} ± {std_f1:.4f}')
print(f'Coverage: {mean_coverage:.4f} ± {std_coverage:.4f}')

NameError: name 'classified_indices' is not defined